# 🚀 KLM — LLaDA Masked Diffusion SLM Training on Google Colab

Notebook này giúp bạn chạy toàn bộ quy trình huấn luyện mô hình ngôn ngữ khuếch tán **KLM (LLaDA Architecture + Hybrid Muon/AdamW + WSD Scheduler)** trên **Google Colab**:

1. **Kiểm tra phần cứng GPU & Mount Google Drive** (lưu trữ bền vững `checkpoints/` và `data/cache/`).
2. **Thiết lập mã nguồn dự án KLM & cài đặt thư viện**.
3. **Tùy chỉnh quy mô mô hình & ngân sách token** (có thể thu nhỏ `budget_scale` để test nhanh trên Colab T4/L4/A100).
4. **Tự huấn luyện bộ BPE Tokenizer (`build_tokenizer.py`)** trực tiếp từ 7 tập dữ liệu tiếng Anh thế hệ mới (tách rời chữ số `0-9` & chuẩn hóa `<|think|>`).
5. **Đóng gói dữ liệu sang các shard nhị phân `.bin` (`prepare_data.py`)**.
6. **Huấn luyện 3 giai đoạn**: `Pretrain` $\rightarrow$ `SFT (Reasoning CoT)` $\rightarrow$ `RL-GRPO`.
7. **Đánh giá Benchmark (`evaluate.py`) & Chạy thử sinh văn bản khuếch tán (`inference.py`)**.

In [ ]:
#@title 1. Kiểm tra GPU & Kết nối Google Drive (Tùy chọn)
import os
import torch

MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_WORKSPACE_DIR = "/content/drive/MyDrive/KLM_Workspace" #@param {type:"string"}

print("=" * 65)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"✅ GPU Detected : {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"✅ BF16 Support : {bf16_ok}")
else:
    print("⚠️ Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> T4 / L4 / A100 GPU.")
print("=" * 65)

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_WORKSPACE_DIR, exist_ok=True)
        print(f"📁 Đã gắn kết Google Drive tại: {DRIVE_WORKSPACE_DIR}")
    except ImportError:
        print("ℹ️ Không chạy trên Google Colab — bỏ qua bước mount Google Drive.")

In [ ]:
#@title 2. Thiết lập thư mục dự án KLM & Cài đặt thư viện
#@markdown Chọn cách đưa mã nguồn `KLM` lên Colab:
#@markdown - **`git_clone`**: Clone từ GitHub repo của bạn.
#@markdown - **`upload_zip`**: Tải tệp `KLM.zip` từ máy tính lên Colab.
#@markdown - **`local_or_drive`**: Đã có sẵn mã nguồn trong thư mục hiện tại hoặc trên Google Drive.

import os
import shutil
import zipfile
from pathlib import Path

SOURCE_MODE = "local_or_drive" #@param ["local_or_drive", "git_clone", "upload_zip"]
GIT_REPO_URL = "" #@param {type:"string"}
PROJECT_DIR = "/content/KLM" #@param {type:"string"}

if Path("config.toml").exists() and Path("train.py").exists():
    PROJECT_DIR = str(Path.cwd())
    print(f"✅ Đang ở sẵn trong thư mục dự án: {PROJECT_DIR}")
else:
    if SOURCE_MODE == "git_clone" and GIT_REPO_URL.strip():
        if not Path(PROJECT_DIR).exists():
            !git clone {GIT_REPO_URL} {PROJECT_DIR}
        os.chdir(PROJECT_DIR)
    elif SOURCE_MODE == "upload_zip":
        from google.colab import files
        print("⬆️ Hãy chọn tệp nén .zip chứa mã nguồn KLM:")
        uploaded = files.upload()
        zip_name = next(iter(uploaded.keys()))
        os.makedirs(PROJECT_DIR, exist_ok=True)
        with zipfile.ZipFile(zip_name, "r") as zf:
            zf.extractall(PROJECT_DIR)
        # Nếu giải nén ra 1 thư mục con KLM/ bên trong PROJECT_DIR
        sub_dirs = [p for p in Path(PROJECT_DIR).iterdir() if p.is_dir() and (p / "config.toml").exists()]
        if sub_dirs:
            PROJECT_DIR = str(sub_dirs[0])
        os.chdir(PROJECT_DIR)
    elif Path(PROJECT_DIR).exists():
        os.chdir(PROJECT_DIR)
    elif "DRIVE_WORKSPACE_DIR" in globals() and (Path(DRIVE_WORKSPACE_DIR) / "config.toml").exists():
        PROJECT_DIR = DRIVE_WORKSPACE_DIR
        os.chdir(PROJECT_DIR)

print(f"📂 Working Directory: {Path.cwd()}")
!pip install -q tokenizers>=0.20.0 safetensors>=0.4.3 datasets>=3.0.0 pyarrow>=15.0.0 transformers tqdm numpy

In [ ]:
#@title 3. Cấu hình Quy mô Mô hình, Ngân sách Token & Lưu trữ Checkpoint
#@markdown Bạn có thể điều chỉnh `TOKEN_BUDGET_SCALE` xuống `0.05` (~15M tokens) hoặc `0.1` (~30M tokens) để test nhanh trên Colab, hoặc giữ `1.0` (300M tokens) để train đầy đủ.

import tomllib
from pathlib import Path

SCALE_PRESET = "200m" #@param ["custom", "10m", "100m", "200m", "500m", "1b"]
VOCAB_SIZE = 32768 #@param {type:"integer"}
TOKEN_BUDGET_SCALE = 0.1 #@param {type:"slider", min:0.02, max:1.0, step:0.02}
MICRO_BATCH_SIZE = 64 #@param {type:"integer"}
GRAD_ACCUM_STEPS = 4 #@param {type:"integer"}
SAVE_TO_DRIVE = True #@param {type:"boolean"}
#@markdown > **Lưu ý T4 GPU**: T4 (16GB) không hỗ trợ `torch.compile(mode='max-autotune')` nhanh như A100/L4; bật `ENABLE_TORCH_COMPILE = False` trên T4 để khởi chạy tức thì.
ENABLE_TORCH_COMPILE = False #@param {type:"boolean"}

# Base token budgets (440M total at scale=1.0: 350M Pretrain + 90M Conversational SFT)
BASE_BUDGETS = {
    "fineweb_edu": 150_000_000,
    "cosmopedia_v2": 100_000_000,
    "tinystories": 40_000_000,
    "finemath_4plus": 35_000_000,
    "opencoder_python": 25_000_000,
    "smoltalk_sft": 45_000_000,
    "ultrachat_sft": 30_000_000,
    "openthoughts3": 15_000_000,
}

ckpt_dir = f"{DRIVE_WORKSPACE_DIR}/checkpoints" if (SAVE_TO_DRIVE and Path("/content/drive/MyDrive").exists()) else "checkpoints"
cache_dir = f"{DRIVE_WORKSPACE_DIR}/data_cache" if (SAVE_TO_DRIVE and Path("/content/drive/MyDrive").exists()) else "data/cache"

# Cập nhật max_tokens trong configs/datasets.toml theo TOKEN_BUDGET_SCALE
datasets_toml_path = Path("configs/datasets.toml")
if datasets_toml_path.exists():
    text = datasets_toml_path.read_text(encoding="utf-8")
    import re
    text = re.sub(r'^cache_dir\s*=\s*".*?"', f'cache_dir = "{cache_dir}"', text, flags=re.MULTILINE)
    for src_name, base_tok in BASE_BUDGETS.items():
        scaled_tok = max(500_000, int(base_tok * TOKEN_BUDGET_SCALE))
        pattern = rf'(name\s*=\s*"{src_name}"[\s\S]*?max_tokens\s*=\s*)[0-9_]+'
        text = re.sub(pattern, rf'\g<1>{scaled_tok:_}', text, count=1)
    datasets_toml_path.write_text(text, encoding="utf-8")

# Ghi cấu hình override vào config.toml
config_override = f"""# Auto-generated Colab overrides
includes = [
    "configs/project.toml",
    "configs/model.toml",
    "configs/training.toml",
    "configs/datasets.toml",
]

[project]
compile = {str(ENABLE_TORCH_COMPILE).lower()}
checkpoint_dir = "{ckpt_dir}"

[model]
scale_preset = "{SCALE_PRESET}"
vocab_size = {VOCAB_SIZE}

[training]
micro_batch_size = {MICRO_BATCH_SIZE}
grad_accum_steps = {GRAD_ACCUM_STEPS}
warmup_ratio = 0.05
stable_ratio = 0.70
decay_ratio = 0.25
mask_ratio_min = 0.15
mask_ratio_max = 0.85
noise_schedule = "cosine"
"""
Path("config.toml").write_text(config_override, encoding="utf-8")

from src.config_loader import load_config
cfg = load_config()
pretrain_total = sum(s.max_tokens for s in cfg.data.active_sources("pretrain"))
sft_total = sum(s.max_tokens for s in cfg.data.active_sources("sft"))
print(f"✅ Đã áp dụng cấu hình! Scale: {cfg.model.scale_preset} | Vocab: {cfg.model.vocab_size}")
print(f"📊 Ngân sách Pretrain : {pretrain_total:,} tokens")
print(f"📊 Ngân sách SFT      : {sft_total:,} tokens")
print(f"💾 Thư mục Checkpoint : {cfg.project.checkpoint_dir}")
print(f"💾 Thư mục Data Cache : {cfg.data.cache_dir}")

In [ ]:
#@title 4. Tự Huấn luyện Bộ BPE Tokenizer (`tokenizer/build_tokenizer.py`)
#@markdown Stream văn bản thực tế từ 7 nguồn dữ liệu trong `configs/datasets.toml` để xây dựng bộ từ vựng Byte-Level BPE riêng (tách rời chữ số `0-9` và gắn thẻ `<|think|>`).

DOCS_PER_SOURCE = 300 #@param {type:"slider", min:50, max:2000, step:50}

!python tokenizer/build_tokenizer.py --docs-per-source {DOCS_PER_SOURCE}

In [ ]:
#@title 5. Tải & Đóng gói Dữ liệu sang Shard Nhị phân `.bin` (`prepare_data.py`)
#@markdown Quá trình này sẽ stream các tập dữ liệu từ Hugging Face, chuẩn hóa hội thoại + thẻ `<|think|>`, mã hóa bằng Rust BPE đa luồng và lưu vào `np.memmap` (`.bin`).

PREPARE_STAGE = "all" #@param ["all", "pretrain", "sft"]
FORCE_RETOKENIZE = False #@param {type:"boolean"}

force_flag = "--force" if FORCE_RETOKENIZE else ""
if PREPARE_STAGE == "all":
    !python prepare_data.py {force_flag}
else:
    !python prepare_data.py --stage {PREPARE_STAGE} {force_flag}

print("\n📋 Trạng thái bộ nhớ đệm (.bin shards):")
!python prepare_data.py --list

In [ ]:
#@title 6. Huấn luyện Giai đoạn 1: LLaDA Masked Diffusion Pre-training
#@markdown Huấn luyện nền tảng trên `FineWeb-Edu`, `FineMath-4plus`, `Cosmopedia-v2`, và `OpenCoder` với lịch trình **WSD + Data Annealing**.

PRETRAIN_STEPS = 3000 #@param {type:"integer"}

!python train.py --stage pretrain --steps {PRETRAIN_STEPS}

In [ ]:
#@title 7. Huấn luyện Giai đoạn 2: Reasoning SFT (OpenThoughts3 + OpenR1-Math + SmolTalk)
#@markdown Tinh chỉnh có giám sát (chỉ mask phần câu trả lời sau `<|assistant|>`, giữ nguyên câu hỏi của `<|user|>`).

SFT_STEPS = 800 #@param {type:"integer"}

!python train.py --stage sft --steps {SFT_STEPS}

In [ ]:
#@title 8. Huấn luyện Giai đoạn 3: RL-GRPO (Tăng cường Suy luận Toán & Logic)
#@markdown Tối ưu hóa chính sách bằng thuật toán **Group Relative Policy Optimization (GRPO)** trên môi trường kiểm chứng tự động.

GRPO_STEPS = 200 #@param {type:"integer"}

!python train.py --stage grpo --steps {GRPO_STEPS}

In [ ]:
#@title 9. Kiểm toán Tham số & Đánh giá Benchmark (`evaluate.py`)
NUM_EVAL_SAMPLES = 20 #@param {type:"integer"}
SHOW_ALL_SCALES = True #@param {type:"boolean"}

scale_flag = "--all-scales" if SHOW_ALL_SCALES else ""
!python evaluate.py --num-samples {NUM_EVAL_SAMPLES} {scale_flag}

In [ ]:
#@title 10. Chạy thử Sinh văn bản Khuếch tán LLaDA (`inference.py`)
PROMPT = "Solve for x: 4 * x + 16 = 64" #@param {type:"string"}
GEN_LENGTH = 64 #@param {type:"slider", min:16, max:256, step:16}
DENOISING_STEPS = 32 #@param {type:"slider", min:4, max:64, step:4}
TEMPERATURE = 0.0 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
SHOW_DENOISING_STEPS = True #@param {type:"boolean"}

step_flag = "--show-steps" if SHOW_DENOISING_STEPS else "--no-stream"
!python inference.py --prompt "{PROMPT}" --gen-length {GEN_LENGTH} --steps {DENOISING_STEPS} --temperature {TEMPERATURE} {step_flag}